In [1]:
from bs4 import BeautifulSoup
import requests
import lxml
import re

import pandas as pd
from datetime import datetime, timedelta
from fake_useragent import UserAgent
ua = UserAgent()

In [2]:
def get_soup(url, timeout=5):
    headers  = {'User-Agent':ua.random}
    try:
        response = requests.get(url,headers=headers)
    except:
        print("FAILED "+ url)
        return 0
    attempts = 0
    while(not response.ok):
            print((url+' failed with code: '+str(response.status_code)))
            if attempts > timeout:
                print(url+' failed with code: '+str(response.status_code))
                return BeautifulSoup('')
            response = requests.get(url)
            attempts += 1
    page = response.text
    soup = BeautifulSoup(page)
    return soup

In [3]:
today = datetime.today() - timedelta(hours=6)
year = str(today.year)
day = str(today.day)
month = '{:02.0f}'.format(today.month)

In [4]:
#######
# January Only
#######
day = int(day)-11
day = str(day)

In [5]:
year

'2022'

In [6]:
month

'03'

In [7]:
day

'6'

In [8]:
url = 'http://sumodb.sumogames.de/Results.aspx?b='+year+month+'&d='+day

In [9]:
soup = get_soup(url)

http://sumodb.sumogames.de/Results.aspx?b=202203&d=6 failed with code: 404
http://sumodb.sumogames.de/Results.aspx?b=202203&d=6 failed with code: 404


KeyboardInterrupt: 

In [43]:
soup = soup.find_all('table',{'class':'tk_table'})[0]

In [44]:
rikishi_east = []
record_east = []
for east in soup.find_all('td',{'class':'tk_east'}):
    text = re.split(r'(\d)',east.find('center').text)
    rikishi_east.append(text[0])
    record_east.append(''.join(text[1:]))

In [45]:
rikishi_west = []
record_west = []
for west in soup.find_all('td',{'class':'tk_west'}):
    text = re.split(r'(\d)',west.find('center').text)
    rikishi_west.append(text[0])
    record_west.append(''.join(text[1:]))

In [46]:
d = {"East":rikishi_east,'East_record':record_east,'West':rikishi_west,'West_record':record_west}

In [47]:
df = pd.DataFrame(d)

In [48]:
df

,East,East_record,West,West_record
0,Terunofuji,2-1,Ichinojo,2-1
1,Shodai,0-3,Ura,0-3
2,Daieisho,2-1,Takakeisho,1-2
3,Takanosho,1-2,Mitakeumi,3-0
4,Meisei,1-2,Abi,2-1
5,Wakatakakage,3-0,Kiribayama,2-1
6,Tamawashi,1-2,Hoshoryu,1-2
7,Onosho,1-2,Takarafuji,1-2
8,Ishiura,1-2,Endo,2-1
9,Takayasu,3-0,Kotonowaka,3-0


In [49]:
df.to_csv('daily_matchups.csv',index=False)

In [50]:
#pd.read_html(url)[2]